<a href="https://colab.research.google.com/github/Ashita829/Context-Aware-Chatbot-Using-LangChain-or-RAG/blob/main/Task4_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community transformers sentence-transformers faiss-cpu gradio -q

from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import pipeline
import gradio as gr
# Read text file
with open("/content/sample.txt", "r", encoding="latin-1") as file:
    text = file.read()

# Split text
splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = splitter.split_text(text)

# Embeddings
embeddings = HuggingFaceEmbeddings()

# Vector DB
db = FAISS.from_texts(texts, embeddings)

# Model
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)
# Chat function
def chatbot(query):

    docs = db.similarity_search(query, k=2)

    context = " ".join([doc.page_content for doc in docs])

    prompt = f"""
    Context:
    {context}

    Question:
    {query}
    """

    response = generator(prompt, max_length=100)

    return response[0]['generated_text']

# Interface
demo = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text",
    title="RAG Chatbot"
)

demo.launch()